# EXPERIMENT_5: Junction-Steered Mask2Former
### Laparoscopic Liver Landmark Detection under Extreme Deformation

This notebook trains and evaluates the **Junction-Steered Mask2Former** architecture:
- Pretrained Swin-Tiny Backbone + MSDeformAttn Pixel Decoder ()
- 4-Query Anatomical Junction Anchor Head ({top}, J_{bottom}, J_{lat\_right}, J_{lat\_left}$)
- Dynamic Query Steering via Cross-Attention to break Spatial Prior Inertia
- Full Multi-Task Training + Patient 40 Visual Diagnostics

In [ ]:
!pip install -q transformers
import os, sys, time, glob, json, cv2
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# NumPy 2.0 compatibility
for _a, _v in [("Inf", np.inf), ("NAN", np.nan), ("NaN", np.nan), ("PINF", np.inf), ("NINF", -np.inf)]:
    if not hasattr(np, _a): setattr(np, _a, _v)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def find_l3d_root():
    candidates = [
        "/kaggle/input/laparoscopic-liver-landmark-dataset/L3D",
        "/kaggle/input/l3d-dataset/L3D",
        "/kaggle/input/l3d/L3D",
        "data/L3D",
        "../data/L3D"
    ]
    for c in candidates:
        if os.path.exists(c):
            print(f"Found L3D dataset at: {c}")
            return os.path.abspath(c)
    for p in glob.glob("/kaggle/input/**/labels", recursive=True):
        parent = os.path.dirname(p)
        if os.path.exists(os.path.join(parent, "Train")):
            print(f"Found L3D dataset at: {parent}")
            return parent
    raise RuntimeError("Could not locate L3D dataset!")

DATA_ROOT = find_l3d_root()
OUT_DIR = "/kaggle/working/results_exp5"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ==============================================================================
# 1. Deterministic Junction Extractor
# ==============================================================================
"""
Deterministic Anatomical Junction Extractor for L3D Dataset.
Extracts the 4 biological landmark junctions from JSON polyline annotations:
  1. J_top (idx 0): Falciform-Silhouette Root (Superior liver boundary)
  2. J_bottom (idx 1): Umbilical Notch (Inferior Falciform meets Ridge)
  3. J_lat_right (idx 2): Right Lateral Tip (Right Ridge meets Right Silhouette)
  4. J_lat_left (idx 3): Left Lateral Tip (Left Ridge meets Left Silhouette)
"""
import numpy as np

JUNCTION_NAMES = ['J_top', 'J_bottom', 'J_lat_right', 'J_lat_left']

def extract_gt_junctions(data, orig_w, orig_h, canvas_size=1024, threshold_px=100.0):
    """
    Extracts normalized coordinates (x, y) in [0, 1]^2 and binary visibility flags (4,)
    for the 4 anatomical junction keypoints from an L3D JSON annotation dict.

    Args:
        data (dict): JSON contents with 'shapes' list.
        orig_w (int): Original image width (handles 4K Patient 32).
        orig_h (int): Original image height.
        canvas_size (int): Working resolution (default 1024).
        threshold_px (float): Distance threshold in canvas pixels to consider curves connected.

    Returns:
        coords_norm (np.ndarray): Shape (4, 2), float32, normalized to [0, 1]^2.
        visibility (np.ndarray): Shape (4,), float32, in {0.0, 1.0}.
        coords_px (np.ndarray): Shape (4, 2), float32, in [0, canvas_size] pixels.
    """
    sx = float(canvas_size) / float(orig_w)
    sy = float(canvas_size) / float(orig_h)

    r_curves, s_curves, f_curves = [], [], []
    for shape in data.get('shapes', []):
        lbl = str(shape.get('label', '')).lower().strip()
        pts = np.array(shape.get('points', []), dtype=np.float32)
        if len(pts) < 2:
            continue
        # Scale to canvas coordinates
        scaled = np.column_stack([pts[:, 0] * sx, pts[:, 1] * sy])
        if lbl.startswith('r') or 'ridge' in lbl or 'rigde' in lbl:
            r_curves.append(scaled)
        elif lbl.startswith('s') or 'sil' in lbl:
            s_curves.append(scaled)
        elif lbl.startswith('f') or 'falc' in lbl or 'lig' in lbl:
            f_curves.append(scaled)

    coords_px = np.zeros((4, 2), dtype=np.float32)
    visibility = np.zeros(4, dtype=np.float32)

    # 1. J_top (Falciform meets Silhouette at superior root)
    best_d = 1e9
    j_top = None
    for fc in f_curves:
        for sc in s_curves:
            for f_pt in [fc[0], fc[-1]]:
                dists = np.linalg.norm(sc - f_pt, axis=1)
                min_idx = np.argmin(dists)
                if dists[min_idx] < best_d:
                    best_d = dists[min_idx]
                    j_top = (f_pt + sc[min_idx]) / 2.0
    if best_d <= threshold_px and j_top is not None:
        coords_px[0] = j_top
        visibility[0] = 1.0

    # 2. J_bottom (Falciform meets Ridge at Umbilical Notch)
    best_d = 1e9
    j_bot = None
    for fc in f_curves:
        for rc in r_curves:
            for f_pt in [fc[0], fc[-1]]:
                dists = np.linalg.norm(rc - f_pt, axis=1)
                min_idx = np.argmin(dists)
                if dists[min_idx] < best_d:
                    best_d = dists[min_idx]
                    j_bot = (f_pt + rc[min_idx]) / 2.0
    if best_d <= threshold_px and j_bot is not None:
        coords_px[1] = j_bot
        visibility[1] = 1.0

    # 3. J_lat_right & J_lat_left (Ridge meets Silhouette at both extremities)
    best_dr, pt_r = 1e9, None
    best_dl, pt_l = 1e9, None

    for rc in r_curves:
        for sc in s_curves:
            # Check ridge endpoints to silhouette curve
            for r_pt in [rc[0], rc[-1]]:
                d_s = np.linalg.norm(sc - r_pt, axis=1)
                idx = np.argmin(d_s)
                d = d_s[idx]
                pt = (r_pt + sc[idx]) / 2.0
                if pt[0] >= canvas_size * 0.45 and d < best_dr:
                    best_dr, pt_r = d, pt
                elif pt[0] < canvas_size * 0.45 and d < best_dl:
                    best_dl, pt_l = d, pt
            # Check silhouette endpoints to ridge curve
            for s_pt in [sc[0], sc[-1]]:
                d_r = np.linalg.norm(rc - s_pt, axis=1)
                idx = np.argmin(d_r)
                d = d_r[idx]
                pt = (s_pt + rc[idx]) / 2.0
                if pt[0] >= canvas_size * 0.45 and d < best_dr:
                    best_dr, pt_r = d, pt
                elif pt[0] < canvas_size * 0.45 and d < best_dl:
                    best_dl, pt_l = d, pt

    if best_dr <= threshold_px and pt_r is not None:
        coords_px[2] = pt_r
        visibility[2] = 1.0

    if best_dl <= threshold_px and pt_l is not None:
        coords_px[3] = pt_l
        visibility[3] = 1.0

    # Normalize coordinates to [0, 1]^2
    coords_norm = coords_px / float(canvas_size)

    return coords_norm, visibility, coords_px


In [ ]:
# ==============================================================================
# 2. Evaluation Metrics (NumPy 2.0 & Pure OpenCV)
# ==============================================================================
"""
Evaluation metrics for EXPERIMENT_5:
  - Macro Dice, IoU, ASSD per class (Ridge, Silhouette, Falciform)
  - Patient 40 diagnostic metrics
  - 4-Junction Mean Absolute Pixel Error (MAPE) & Visibility Accuracy
  - Pure OpenCV & NumPy implementation (zero external C-extension dependencies)
"""
import cv2
import numpy as np
import torch

# NumPy 2.0 compatibility
for _a, _v in [('Inf', np.inf), ('NAN', np.nan), ('NaN', np.nan), ('PINF', np.inf), ('NINF', -np.inf)]:
    if not hasattr(np, _a):
        setattr(np, _a, _v)

def compute_dice(pred_bin, target_bin, eps=1e-6):
    pred_b = (pred_bin > 0).astype(bool)
    target_b = (target_bin > 0).astype(bool)
    intersection = np.logical_and(pred_b, target_b).sum()
    total = pred_b.sum() + target_b.sum()
    if total == 0:
        return 1.0
    return float(2.0 * intersection / (total + eps))

def compute_iou(pred_bin, target_bin, eps=1e-6):
    pred_b = (pred_bin > 0).astype(bool)
    target_b = (target_bin > 0).astype(bool)
    intersection = np.logical_and(pred_b, target_b).sum()
    union = np.logical_or(pred_b, target_b).sum()
    if union == 0:
        return 1.0
    return float(intersection / (union + eps))

def compute_assd_fast(pred_bin, target_bin, max_penalty=80.0):
    """
    Computes Average Symmetric Surface Distance (ASSD) using OpenCV distanceTransform.
    Guaranteed zero external dependency issues.
    """
    p_b = (pred_bin > 0).astype(np.uint8)
    t_b = (target_bin > 0).astype(np.uint8)
    
    if p_b.sum() == 0 and t_b.sum() == 0:
        return 0.0
    if p_b.sum() == 0 or t_b.sum() == 0:
        return max_penalty
        
    try:
        # Extract 1-pixel boundary edges
        contours_p, _ = cv2.findContours(p_b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        contours_t, _ = cv2.findContours(t_b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        
        edge_p = np.zeros_like(p_b)
        edge_t = np.zeros_like(t_b)
        cv2.drawContours(edge_p, contours_p, -1, 1, 1)
        cv2.drawContours(edge_t, contours_t, -1, 1, 1)
        
        if edge_p.sum() == 0 or edge_t.sum() == 0:
            return max_penalty
            
        dist_t = cv2.distanceTransform((1 - edge_t).astype(np.uint8), cv2.DIST_L2, 3)
        dist_p = cv2.distanceTransform((1 - edge_p).astype(np.uint8), cv2.DIST_L2, 3)
        
        d_p2t = np.mean(dist_t[edge_p > 0])
        d_t2p = np.mean(dist_p[edge_t > 0])
        
        return float((d_p2t + d_t2p) / 2.0)
    except Exception:
        return max_penalty

def evaluate_frame_metrics(pred_map, target_map, pred_j_coords=None, gt_j_coords=None, gt_j_vis=None, canvas_size=1024):
    """
    Computes all semantic and junction metrics for a single 1024x1024 frame.
    """
    c_dices = []
    c_ious = []
    c_assds = []
    
    # Classes: 1=Ridge, 2=Sil, 3=Falc
    for c in [1, 2, 3]:
        p_c = (pred_map == c)
        t_c = (target_map == c)
        c_dices.append(compute_dice(p_c, t_c))
        c_ious.append(compute_iou(p_c, t_c))
        c_assds.append(compute_assd_fast(p_c, t_c))
        
    macro_dice = float(np.mean(c_dices))
    macro_iou = float(np.mean(c_ious))
    macro_assd = float(np.mean(c_assds))
    fg_dice = compute_dice(pred_map > 0, target_map > 0)
    
    res = {
        'macro_dice': macro_dice,
        'macro_iou': macro_iou,
        'macro_assd': macro_assd,
        'ridge_dice': c_dices[0],
        'sil_dice': c_dices[1],
        'falc_dice': c_dices[2],
        'fg_dice': fg_dice,
        'ridge_assd': c_assds[0],
        'sil_assd': c_assds[1],
        'falc_assd': c_assds[2]
    }
    
    # Junction metrics (if available)
    if pred_j_coords is not None and gt_j_coords is not None and gt_j_vis is not None:
        j_errs = []
        names = ['j_top_err_px', 'j_bot_err_px', 'j_lat_r_err_px', 'j_lat_l_err_px']
        for k in range(4):
            if gt_j_vis[k] > 0.5:
                err_px = float(np.linalg.norm((pred_j_coords[k] - gt_j_coords[k]) * float(canvas_size)))
                j_errs.append(err_px)
                res[names[k]] = err_px
            else:
                res[names[k]] = None
        res['mean_j_err_px'] = float(np.mean(j_errs)) if len(j_errs) > 0 else None
        
    return res


In [ ]:
# ==============================================================================
# 3. L3D Dataset Reader (Patient 32 4K Preserved)
# ==============================================================================
"""
L3D PyTorch Dataset for EXPERIMENT_5 (Junction-Steered Mask2Former).
Provides:
  - RGB images (1024x1024, normalized with ImageNet stats)
  - Dense 3-class segmentation masks (0=BG, 1=Ridge, 2=Sil, 3=Falc)
  - Ground-truth 4-junction coordinates in [0, 1]^2
  - Ground-truth 4-junction binary visibility flags
  - Automatic environment resolution (local macOS vs. Kaggle CUDA)
"""
import os
import glob
import json
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path

from experiments.EXPERIMENT_5.utils.junction_extractor import extract_gt_junctions

# Standard ImageNet statistics
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def resolve_l3d_root(candidate_root=None):
    """
    Auto-detects the L3D dataset root path across local macOS, Linux, and Kaggle.
    """
    candidates = [
        candidate_root,
        "data/L3D",
        "../data/L3D",
        "../../data/L3D",
        "/kaggle/input/laparoscopic-liver-landmark-dataset/L3D",
        "/kaggle/input/l3d-dataset/L3D",
        "/kaggle/input/l3d/L3D",
        "/data/khoalq/data/L3D"
    ]
    for c in candidates:
        if c and os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath("data/L3D")

class L3DDataset(Dataset):
    def __init__(self, split='Train', data_dir=None, image_size=1024, stroke_width=35):
        super().__init__()
        self.split = split
        self.image_size = image_size
        self.stroke_width = stroke_width
        self.root_dir = resolve_l3d_root(data_dir)
        
        split_dir = os.path.join(self.root_dir, split)
        self.img_dir = os.path.join(split_dir, 'images')
        self.lbl_dir = os.path.join(split_dir, 'labels')
        
        self.json_files = sorted(glob.glob(os.path.join(self.lbl_dir, '*.json')))
        if len(self.json_files) == 0:
            raise RuntimeError(f"No JSON annotation files found in: {self.lbl_dir}")
            
        print(f"[{split}] Loaded {len(self.json_files)} frames from: {split_dir}")

    def __len__(self):
        return len(self.json_files)

    def __getitem__(self, idx):
        json_path = self.json_files[idx]
        stem = Path(json_path).stem
        img_path = os.path.join(self.img_dir, f"{stem}.jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir, f"{stem}.png")
            
        # 1. Load image
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            raise FileNotFoundError(f"Failed to read image at: {img_path}")
            
        orig_h, orig_w = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # Resize image to canvas_size
        if orig_w != self.image_size or orig_h != self.image_size:
            img_resized = cv2.resize(img_rgb, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        else:
            img_resized = img_rgb
            
        # Normalize pixel values
        norm_img = (img_resized.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        pixel_values = torch.from_numpy(norm_img).permute(2, 0, 1).float() # (3, H, W)
        
        # 2. Parse JSON annotations
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        # Handle Patient 32 4K canvas resolution properly
        data_w = data.get('imageWidth', orig_w)
        data_h = data.get('imageHeight', orig_h)
        sx = float(self.image_size) / float(data_w)
        sy = float(self.image_size) / float(data_h)
        
        # 3. Create dense raster mask
        mask = np.zeros((self.image_size, self.image_size), dtype=np.uint8)
        
        # Draw in order: Ridge (1), Silhouette (2), Falciform (3)
        # Note typo tolerance: 'rigde'
        shapes = data.get('shapes', [])
        
        for shape in shapes:
            lbl = str(shape.get('label', '')).lower().strip()
            pts = np.array(shape.get('points', []), dtype=np.float32)
            if len(pts) < 2:
                continue
                
            pts[:, 0] *= sx
            pts[:, 1] *= sy
            pts_int = np.round(pts).astype(np.int32).reshape((-1, 1, 2))
            
            if lbl.startswith('r') or 'ridge' in lbl or 'rigde' in lbl:
                cv2.polylines(mask, [pts_int], isClosed=False, color=1, thickness=self.stroke_width, lineType=cv2.LINE_AA)
            elif lbl.startswith('s') or 'sil' in lbl:
                cv2.polylines(mask, [pts_int], isClosed=False, color=2, thickness=self.stroke_width, lineType=cv2.LINE_AA)
            elif lbl.startswith('f') or 'falc' in lbl or 'lig' in lbl:
                cv2.polylines(mask, [pts_int], isClosed=False, color=3, thickness=self.stroke_width, lineType=cv2.LINE_AA)

        mask_tensor = torch.from_numpy(mask).long() # (H, W)
        
        # 4. Extract 4 GT Anatomical Junctions
        coords_norm, vis, _ = extract_gt_junctions(data, data_w, data_h, canvas_size=self.image_size)
        junction_coords = torch.from_numpy(coords_norm).float() # (4, 2)
        junction_vis = torch.from_numpy(vis).float()           # (4,)
        
        is_p40 = ('patient_40' in stem.lower() or '_40_' in stem.lower())
        
        return {
            'pixel_values': pixel_values,
            'mask': mask_tensor,
            'junction_coords': junction_coords,
            'junction_vis': junction_vis,
            'filename': f"{stem}.jpg",
            'is_patient_40': is_p40
        }


In [ ]:
# ==============================================================================
# 4. Junction Anchor Head & Steering Block
# ==============================================================================
"""
Junction Anchor Head for EXPERIMENT_5.
Extracts 4 biological landmark vectors and predicts (x, y) coordinates + visibility flags:
  0: J_top (Falc-Sil Root)
  1: J_bottom (Umbilical Notch)
  2: J_lat_right (Right Lateral Tip)
  3: J_lat_left (Left Lateral Tip)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class JunctionDecoderLayer(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout)
        )
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, context):
        """
        queries: (B, 4, C)
        context: (B, S, C)
        """
        # 1. Self-attention among the 4 junction nodes (anatomical graph reasoning)
        q_norm = self.norm1(queries)
        q_self, _ = self.self_attn(q_norm, q_norm, q_norm)
        queries = queries + self.dropout(q_self)
        
        # 2. Cross-attention from context feature map
        q_norm = self.norm2(queries)
        q_cross, _ = self.cross_attn(query=q_norm, key=context, value=context)
        queries = queries + self.dropout(q_cross)
        
        # 3. Feedforward
        queries = queries + self.ffn(self.norm3(queries))
        return queries

class JunctionAnchorHead(nn.Module):
    def __init__(self, embed_dim=256, num_layers=2, num_heads=8):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_junctions = 4
        
        # 4 Learnable Anatomical Query Embeddings
        self.junction_queries = nn.Parameter(torch.randn(1, 4, embed_dim) * 0.02)
        
        # 2-Layer Transformer Decoder
        self.layers = nn.ModuleList([
            JunctionDecoderLayer(embed_dim=embed_dim, num_heads=num_heads)
            for _ in range(num_layers)
        ])
        
        # Coordinate Regression Head: (B, 4, 2) in [0, 1]^2
        self.coord_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )
        
        # Visibility Prediction Head: (B, 4) binary logit
        self.vis_head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        
        # Anatomical Prior Biases (logits for initial coordinate sigmoid)
        # [0]: Top (0.50, 0.20), [1]: Bot (0.50, 0.70), [2]: Lat_R (0.85, 0.40), [3]: Lat_L (0.15, 0.60)
        priors = torch.tensor([
            [0.50, 0.20],
            [0.50, 0.70],
            [0.85, 0.40],
            [0.15, 0.60]
        ], dtype=torch.float32)
        # logit(p) = log(p / (1 - p))
        prior_logits = torch.log(priors / (1.0 - priors + 1e-6))
        self.register_buffer('prior_logits', prior_logits)

    def forward(self, feature_map):
        """
        Args:
            feature_map (Tensor): (B, 256, H, W) from Pixel Decoder (e.g. stride 16).
        Returns:
            coords (Tensor): (B, 4, 2) normalized to [0, 1]^2
            vis_logits (Tensor): (B, 4) unnormalized visibility logits
            features (Tensor): (B, 4, 256) junction feature tokens
        """
        B, C, H, W = feature_map.shape
        context = feature_map.flatten(2).transpose(1, 2) # (B, H*W, C)
        
        queries = self.junction_queries.expand(B, -1, -1) # (B, 4, C)
        
        for layer in self.layers:
            queries = layer(queries, context)
            
        # Predict coordinates with anatomical prior bias
        coord_deltas = self.coord_head(queries) # (B, 4, 2)
        coords = torch.sigmoid(coord_deltas + self.prior_logits.unsqueeze(0))
        
        # Predict visibility logits
        vis_logits = self.vis_head(queries).squeeze(-1) # (B, 4)
        
        return coords, vis_logits, queries


In [ ]:
# ==============================================================================
# 5. Junction-Steered Mask2Former Architecture
# ==============================================================================
"""
Junction-Steered Mask2Former Model for EXPERIMENT_5.
Combines:
  1. Pretrained Swin-Tiny Backbone + MSDeformAttn Pixel Decoder (from HF Mask2Former)
  2. 4-Query Junction Anchor Head (predicting Top, Bottom, Lat_R, Lat_L)
  3. Dynamic Junction Cross-Attention Query Steering Block
  4. 9-Layer Mask2Former Transformer Decoder
"""
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure workspace root is on sys.path
_WORKSPACE_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '../../..'))
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)

from experiments.EXPERIMENT_5.models.junction_head import JunctionAnchorHead

try:
    from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

class JunctionQuerySteering(nn.Module):
    """
    Dynamic Query Steering Block:
    Allows 100 Mask2Former queries to cross-attend to the 4 junction anchor vectors.
    """
    def __init__(self, embed_dim=256, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.proj_q = nn.Linear(embed_dim, embed_dim)
        self.proj_k = nn.Linear(embed_dim, embed_dim)
        self.proj_v = nn.Linear(embed_dim, embed_dim)
        
        # Learnable gating scalar initialized to 0.1 for smooth gradient highway
        self.gate = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward_transformer_decoder(self, multi_scale_features, mask_features, steered_query_feat):
        tm = self.m2f.model.transformer_module
        multi_stage_features = []
        multi_stage_positional_embeddings = []
        size_list = []

        for i in range(tm.num_feature_levels):
            size_list.append(multi_scale_features[i].shape[-2:])
            multi_stage_positional_embeddings.append(
                tm.position_embedder(
                    multi_scale_features[i].shape, multi_scale_features[i].device, multi_scale_features[i].dtype, None
                ).flatten(2)
            )
            multi_stage_features.append(
                tm.input_projections[i](multi_scale_features[i]).flatten(2)
                + tm.level_embed.weight[i][None, :, None]
            )
            multi_stage_positional_embeddings[-1] = multi_stage_positional_embeddings[-1].permute(2, 0, 1)
            multi_stage_features[-1] = multi_stage_features[-1].permute(2, 0, 1)

        _, batch_size, _ = multi_stage_features[0].shape

        query_embeddings = tm.queries_embedder.weight.unsqueeze(1).repeat(1, batch_size, 1)
        query_features = steered_query_feat.permute(1, 0, 2)

        decoder_output = tm.decoder(
            inputs_embeds=query_features,
            multi_stage_positional_embeddings=multi_stage_positional_embeddings,
            pixel_embeddings=mask_features,
            encoder_hidden_states=multi_stage_features,
            query_position_embeddings=query_embeddings,
            feature_size_list=size_list,
            output_hidden_states=False,
            output_attentions=False,
            return_dict=True,
        )
        return decoder_output

    def forward(self, queries, junction_features, junction_coords=None):
        """
        Args:
            queries (Tensor): (B, 100, C) base query features
            junction_features (Tensor): (B, 4, C) junction anchor tokens
            junction_coords (Tensor, optional): (B, 4, 2) normalized coordinates
        Returns:
            steered_queries (Tensor): (B, 100, C)
            attn_weights (Tensor): (B, 100, 4)
        """
        # Linear projections
        q = self.proj_q(queries)
        k = self.proj_k(junction_features)
        v = self.proj_v(junction_features)
        
        # Cross-attention: queries attend to 4 junctions
        delta_q, attn_weights = self.cross_attn(query=q, key=k, value=v)
        
        # Gated residual update: Q_steered = Norm(Q + gate * delta_Q)
        steered_queries = self.norm(queries + self.gate * self.dropout(delta_q))
        
        return steered_queries, attn_weights

class JunctionSteeredMask2Former(nn.Module):
    """
    Integrated Junction-Steered Mask2Former.
    """
    def __init__(self, model_name="facebook/mask2former-swin-tiny-ade-semantic", num_labels=4):
        super().__init__()
        if not HAS_TRANSFORMERS:
            raise ImportError("HuggingFace 'transformers' library is required. Please install it or activate your PyTorch CUDA env.")
            
        print(f"📦 Building JunctionSteeredMask2Former with base: '{model_name}'...")
        self.m2f = Mask2FormerForUniversalSegmentation.from_pretrained(
            model_name,
            num_labels=num_labels,
            ignore_mismatched_sizes=True
        )
        
        embed_dim = 256
        
        # 1. Junction Anchor Head (4 biological queries)
        self.junction_head = JunctionAnchorHead(embed_dim=embed_dim, num_layers=2, num_heads=8)
        
        # 2. Query Steering Block
        self.query_steering = JunctionQuerySteering(embed_dim=embed_dim, num_heads=8)
        
    def forward_transformer_decoder(self, multi_scale_features, mask_features, steered_query_feat):
        tm = self.m2f.model.transformer_module
        multi_stage_features = []
        multi_stage_positional_embeddings = []
        size_list = []

        for i in range(tm.num_feature_levels):
            size_list.append(multi_scale_features[i].shape[-2:])
            multi_stage_positional_embeddings.append(
                tm.position_embedder(
                    multi_scale_features[i].shape, multi_scale_features[i].device, multi_scale_features[i].dtype, None
                ).flatten(2)
            )
            multi_stage_features.append(
                tm.input_projections[i](multi_scale_features[i]).flatten(2)
                + tm.level_embed.weight[i][None, :, None]
            )
            multi_stage_positional_embeddings[-1] = multi_stage_positional_embeddings[-1].permute(2, 0, 1)
            multi_stage_features[-1] = multi_stage_features[-1].permute(2, 0, 1)

        _, batch_size, _ = multi_stage_features[0].shape

        query_embeddings = tm.queries_embedder.weight.unsqueeze(1).repeat(1, batch_size, 1)
        query_features = steered_query_feat.permute(1, 0, 2)

        decoder_output = tm.decoder(
            inputs_embeds=query_features,
            multi_stage_positional_embeddings=multi_stage_positional_embeddings,
            pixel_embeddings=mask_features,
            encoder_hidden_states=multi_stage_features,
            query_position_embeddings=query_embeddings,
            feature_size_list=size_list,
            output_hidden_states=False,
            output_attentions=False,
            return_dict=True,
        )
        return decoder_output

    def forward(self, pixel_values, mask_labels=None, class_labels=None):
        """
        Forward pass with dynamic query steering.
        Args:
            pixel_values (Tensor): (B, 3, H, W)
            mask_labels (list of Tensors, optional): GT binary masks per image for training
            class_labels (list of Tensors, optional): GT class IDs per image for training
        Returns:
            dict containing:
                - 'loss': total loss (if mask_labels is provided)
                - 'masks_queries_logits': (B, 100, H/4, W/4)
                - 'class_queries_logits': (B, 100, num_classes + 1)
                - 'pred_junction_coords': (B, 4, 2) in [0, 1]^2
                - 'pred_junction_vis': (B, 4) visibility logits
                - 'junction_attn_weights': (B, 100, 4)
                - 'm2f_loss': standard Hungarian loss (if training)
        """
        B = pixel_values.shape[0]
        
        # 1. Swin-Tiny Backbone + MSDeformAttn Pixel Decoder
        pixel_level_outputs = self.m2f.model.pixel_level_module(pixel_values, output_hidden_states=True)
        # multi_scale_features has 3 tensors: [stride 32, stride 16, stride 8]
        multi_scale_features = list(pixel_level_outputs.decoder_hidden_states)
        mask_features = pixel_level_outputs.decoder_last_hidden_state # (B, 256, H/4, W/4)
        
        # 2. Extract 4 Anatomical Junctions from stride-16 features
        stride_16_feat = multi_scale_features[1] # (B, 256, H_16, W_16)
        pred_j_coords, pred_j_vis, j_features = self.junction_head(stride_16_feat)
        
        # 3. Dynamic Query Steering:
        # Base query features from Mask2Former
        tm = self.m2f.model.transformer_module
        base_query_feat = tm.queries_features.weight.unsqueeze(0).repeat(B, 1, 1) # (B, 100, 256)
        
        # Steer the queries with the 4 junction vectors
        steered_query_feat, j_attn_weights = self.query_steering(base_query_feat, j_features, pred_j_coords)
        
        # 4. Forward through Mask2Former Transformer Decoder with steered queries
        # Temporarily patch queries_features during this forward pass
        orig_queries_features = tm.queries_features
        class DynamicQueryFeatures(nn.Module):
            def __init__(self, steered_tensor):
                super().__init__()
                self.steered = steered_tensor
            def forward_transformer_decoder(self, multi_scale_features, mask_features, steered_query_feat):
        tm = self.m2f.model.transformer_module
        multi_stage_features = []
        multi_stage_positional_embeddings = []
        size_list = []

        for i in range(tm.num_feature_levels):
            size_list.append(multi_scale_features[i].shape[-2:])
            multi_stage_positional_embeddings.append(
                tm.position_embedder(
                    multi_scale_features[i].shape, multi_scale_features[i].device, multi_scale_features[i].dtype, None
                ).flatten(2)
            )
            multi_stage_features.append(
                tm.input_projections[i](multi_scale_features[i]).flatten(2)
                + tm.level_embed.weight[i][None, :, None]
            )
            multi_stage_positional_embeddings[-1] = multi_stage_positional_embeddings[-1].permute(2, 0, 1)
            multi_stage_features[-1] = multi_stage_features[-1].permute(2, 0, 1)

        _, batch_size, _ = multi_stage_features[0].shape

        query_embeddings = tm.queries_embedder.weight.unsqueeze(1).repeat(1, batch_size, 1)
        query_features = steered_query_feat.permute(1, 0, 2)

        decoder_output = tm.decoder(
            inputs_embeds=query_features,
            multi_stage_positional_embeddings=multi_stage_positional_embeddings,
            pixel_embeddings=mask_features,
            encoder_hidden_states=multi_stage_features,
            query_position_embeddings=query_embeddings,
            feature_size_list=size_list,
            output_hidden_states=False,
            output_attentions=False,
            return_dict=True,
        )
        return decoder_output

    def forward(self, *args, **kwargs):
                return self.steered
                
        # Run standard transformer module forward
        # In HuggingFace, transformer_module forward expects (multi_scale_features, mask_features)
        # We temporarily hook the query features
        old_weight = tm.queries_features.weight
        # Steered mean across batch as weight, or pass per-sample into decoder:
        # HuggingFace transformer_module:
        # query_features = self.queries_features.weight.unsqueeze(0).repeat(batch_size, 1, 1)
        # We hook tm.queries_features:
        tm.queries_features = DynamicQueryFeatures(steered_query_feat)
        
        try:
            decoder_outputs = tm(
                multi_scale_features,
                mask_features,
                output_hidden_states=False,
                output_attentions=False
            )
        finally:
            tm.queries_features = orig_queries_features # Restore
            
        # Extract class & mask logits
        class_queries_logits = self.m2f.class_predictor(decoder_outputs[0]) # (B, 100, num_classes + 1)
        mask_embeddings = self.m2f.mask_embedder(decoder_outputs[0])       # (B, 100, 256)
        # Dot product with mask_features: (B, 100, 256) x (B, 256, H/4, W/4) -> (B, 100, H/4, W/4)
        masks_queries_logits = torch.einsum("bqc,bchw->bqhw", mask_embeddings, mask_features)
        
        output = {
            'masks_queries_logits': masks_queries_logits,
            'class_queries_logits': class_queries_logits,
            'pred_junction_coords': pred_j_coords,
            'pred_junction_vis': pred_j_vis,
            'junction_attn_weights': j_attn_weights
        }
        
        # 5. Compute Hungarian Matching Loss if GT labels are provided
        if mask_labels is not None and class_labels is not None:
            # Build loss using Mask2Former's built-in criterion
            loss_dict = self.m2f.criterion(
                masks_queries_logits=masks_queries_logits,
                class_queries_logits=class_queries_logits,
                mask_labels=mask_labels,
                class_labels=class_labels
            )
            # Weighted sum according to HuggingFace weight_dict
            weight_dict = self.m2f.criterion.weight_dict
            m2f_loss = sum(loss_dict[k] * weight_dict[k] for k in loss_dict.keys() if k in weight_dict)
            output['m2f_loss'] = m2f_loss
            output['m2f_loss_dict'] = loss_dict
            
        return output

def load_junction_steered_model(checkpoint_path=None, model_name="facebook/mask2former-swin-tiny-ade-semantic", device="cpu"):
    """
    Helper to instantiate and load checkpoint weights.
    """
    model = JunctionSteeredMask2Former(model_name=model_name, num_labels=4)
    if checkpoint_path and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
        model.load_state_dict(state_dict)
        print(f"✅ Loaded checkpoint from: {checkpoint_path}")
    return model.to(device)


In [ ]:
# ==============================================================================
# 6. Multi-Task Loss
# ==============================================================================
"""
Multi-Task Loss for Junction-Steered Mask2Former (EXPERIMENT_5).
Combines:
  1. Mask2Former Hungarian Matching Loss (Classification Focal + Mask BCE + Mask Dice)
  2. Visibility-Masked Coordinate Smooth-L1 Loss on 4 Biological Junctions
  3. Binary Cross-Entropy on Junction Visibility Flags
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class JunctionSteeredLoss(nn.Module):
    def __init__(self, lambda_m2f=1.0, lambda_coord=5.0, lambda_vis=1.0):
        super().__init__()
        self.lambda_m2f = lambda_m2f
        self.lambda_coord = lambda_coord
        self.lambda_vis = lambda_vis
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(self, model_outputs, gt_junction_coords, gt_junction_vis):
        """
        Args:
            model_outputs (dict): Output dict from JunctionSteeredMask2Former containing:
                - 'm2f_loss': scalar Hungarian loss
                - 'pred_junction_coords': (B, 4, 2) in [0, 1]^2
                - 'pred_junction_vis': (B, 4) visibility logits
            gt_junction_coords (Tensor): (B, 4, 2) normalized GT coordinates
            gt_junction_vis (Tensor): (B, 4) binary GT visibility flags {0, 1}
        Returns:
            total_loss (Tensor): scalar
            loss_dict (dict): component breakdowns
        """
        device = gt_junction_coords.device
        pred_coords = model_outputs['pred_junction_coords'] # (B, 4, 2)
        pred_vis_logits = model_outputs['pred_junction_vis'] # (B, 4)
        
        # 1. Mask2Former Base Loss
        m2f_loss = model_outputs.get('m2f_loss', torch.tensor(0.0, device=device))
        
        # 2. Junction Visibility Loss (BCE)
        vis_loss = self.bce_loss(pred_vis_logits, gt_junction_vis)
        
        # 3. Visibility-Masked Coordinate Loss (Smooth L1)
        # Coordinate loss is only computed for junctions visible in GT
        vis_mask = (gt_junction_vis > 0.5) # (B, 4)
        if vis_mask.sum() > 0:
            # Expand mask for 2D coords: (B, 4, 2)
            coord_mask = vis_mask.unsqueeze(-1).expand_as(pred_coords)
            diff = F.smooth_l1_loss(
                pred_coords[coord_mask],
                gt_junction_coords[coord_mask],
                beta=0.02,
                reduction='mean'
            )
            coord_loss = diff
        else:
            coord_loss = torch.tensor(0.0, device=device)
            
        # Total Weighted Multi-Task Loss
        total_loss = (self.lambda_m2f * m2f_loss +
                      self.lambda_coord * coord_loss +
                      self.lambda_vis * vis_loss)
                      
        loss_dict = {
            'loss': total_loss.item(),
            'm2f_loss': m2f_loss.item() if isinstance(m2f_loss, torch.Tensor) else m2f_loss,
            'coord_loss': coord_loss.item() if isinstance(coord_loss, torch.Tensor) else coord_loss,
            'vis_loss': vis_loss.item() if isinstance(vis_loss, torch.Tensor) else vis_loss
        }
        
        return total_loss, loss_dict


In [ ]:
# ==============================================================================
# 7. Patient 40 Diagnostics & Evaluation Loop
# ==============================================================================
"""
Standalone Evaluation Script for EXPERIMENT_5 (Junction-Steered Mask2Former).
Evaluates:
  - Validation Split (122 frames)
  - Unseen Test Split (109 frames)
Produces:
  - metrics_summary.json
  - val_predictions.csv
  - test_predictions.csv
  - patient_40_diagnostics/ (4-panel diagnostic visualizations for Patient 40 only)
"""
import os
import sys
import time
import json
import argparse
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import DataLoader
from pathlib import Path

# NumPy 2.0 monkeypatch
if not hasattr(np, 'Inf'):
    np.Inf = np.inf
    np.PINF = np.inf
    np.NINF = -np.inf

# Ensure workspace root is on sys.path
_WORKSPACE_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '../../..'))
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)

from experiments.EXPERIMENT_5.utils.dataset import L3DDataset, IMAGENET_MEAN, IMAGENET_STD
from experiments.EXPERIMENT_5.utils.metrics import evaluate_frame_metrics
from experiments.EXPERIMENT_5.models.junction_steered_mask2former import load_junction_steered_model

# Canonical colors
COLOR_MAP_RGB = {
    0: (0, 0, 0),       # BG
    1: (34, 197, 94),   # Ridge: Green (#22c55e)
    2: (239, 68, 68),   # Silhouette: Red (#ef4444)
    3: (59, 130, 246)   # Falciform: Blue (#3b82f6)
}

JUNCTION_COLORS_BGR = {
    0: (0, 215, 255), # Top: Gold
    1: (255, 255, 0), # Bottom: Cyan
    2: (255, 0, 255), # Lat_Right: Magenta
    3: (0, 140, 255)  # Lat_Left: Orange
}

def rasterize_class_map(masks_queries_logits, class_queries_logits, canvas_size=1024):
    """
    Standard Mask2Former post-processing:
    Argmax over query probabilities multiplied by sigmoid mask probabilities.
    """
    masks_queries_logits = masks_queries_logits.float()
    class_queries_logits = class_queries_logits.float()
    # masks_queries_logits: (100, H/4, W/4)
    # class_queries_logits: (100, num_classes + 1)
    
    # 1. Resize mask logits to canvas_size
    masks = F_interpolate = torch.nn.functional.interpolate(
        masks_queries_logits.unsqueeze(0),
        size=(canvas_size, canvas_size),
        mode='bilinear',
        align_corners=False
    ).squeeze(0).sigmoid() # (100, H, W)
    
    # 2. Query classification probabilities
    cls_probs = torch.softmax(class_queries_logits, dim=-1) # (100, 5), last is BG
    
    # Exclude background class (index 4)
    fg_cls_probs = cls_probs[:, :4] # (100, 4)
    
    # Multiply: (100, 4, 1, 1) * (100, 1, H, W) -> (100, 4, H, W)
    # Sem_prob[c, h, w] = sum_q (cls_prob[q, c] * mask_prob[q, h, w])
    sem_probs = torch.einsum("qc,qhw->chw", fg_cls_probs, masks) # (4, H, W)
    
    # Thresholding
    pred_map = sem_probs.argmax(dim=0).cpu().numpy().astype(np.int64) # (H, W)
    # Background suppression where probability is very low
    max_prob = sem_probs.max(dim=0)[0].cpu().numpy()
    pred_map[max_prob < 0.25] = 0
    
    return pred_map

def render_patient_40_diagnostic(orig_rgb_norm, gt_mask, pred_map, pred_j_coords, gt_j_coords, gt_j_vis, out_path, metrics):
    """
    Renders a 4-panel diagnostic montage (1024x1024 x 4 = 2048x2048) for Patient 40 only:
      [ Panel 1: Input RGB ]               [ Panel 2: Ground Truth + GT Junctions ]
      [ Panel 3: Prediction + Pred Juncs ]  [ Panel 4: Error Overlay (TP, FP, FN)   ]
    """
    # Unnormalize RGB
    rgb = (orig_rgb_norm.transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN) * 255.0
    rgb = np.clip(rgb, 0, 255).astype(np.uint8)
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    
    H, W = bgr.shape[:2]
    
    # Panel 2: GT Overlay
    p2 = bgr.copy()
    for c_id in [1, 2, 3]:
        mask_c = (gt_mask == c_id)
        if mask_c.any():
            col = COLOR_MAP_RGB[c_id][::-1] # RGB to BGR
            p2[mask_c] = (p2[mask_c] * 0.4 + np.array(col) * 0.6).astype(np.uint8)
    # Draw GT Junctions
    for k in range(4):
        if gt_j_vis[k] > 0.5:
            pt = (gt_j_coords[k] * float(W)).astype(int)
            cv2.circle(p2, tuple(pt), 14, (0, 0, 0), -1)
            cv2.circle(p2, tuple(pt), 10, JUNCTION_COLORS_BGR[k], -1)
            cv2.circle(p2, tuple(pt), 3, (255, 255, 255), -1)
            
    # Panel 3: Pred Overlay
    p3 = bgr.copy()
    for c_id in [1, 2, 3]:
        mask_c = (pred_map == c_id)
        if mask_c.any():
            col = COLOR_MAP_RGB[c_id][::-1]
            p3[mask_c] = (p3[mask_c] * 0.4 + np.array(col) * 0.6).astype(np.uint8)
    # Draw Pred Junctions
    for k in range(4):
        pt = (pred_j_coords[k] * float(W)).astype(int)
        cv2.circle(p3, tuple(pt), 14, (0, 0, 0), -1)
        cv2.circle(p3, tuple(pt), 10, JUNCTION_COLORS_BGR[k], -1)
        cv2.circle(p3, tuple(pt), 3, (0, 0, 0), -1)
        
    # Panel 4: Error Map (TP=Green, FP=Cyan, FN=Red)
    p4 = bgr.copy()
    tp = (pred_map > 0) & (gt_mask > 0)
    fp = (pred_map > 0) & (gt_mask == 0)
    fn = (pred_map == 0) & (gt_mask > 0)
    
    p4[tp] = (p4[tp] * 0.3 + np.array([0, 255, 0]) * 0.7).astype(np.uint8)   # TP: Green
    p4[fp] = (p4[fp] * 0.3 + np.array([255, 255, 0]) * 0.7).astype(np.uint8) # FP: Cyan
    p4[fn] = (p4[fn] * 0.3 + np.array([0, 0, 255]) * 0.7).astype(np.uint8)   # FN: Red
    
    # Stitch 2x2 grid
    top_row = np.hstack([bgr, p2])
    bot_row = np.hstack([p3, p4])
    montage = np.vstack([top_row, bot_row])
    
    # Add title headers
    cv2.putText(montage, "1. Input Laparoscopic Frame", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.putText(montage, "2. Ground Truth + GT Junctions", (W + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.putText(montage, f"3. Prediction (Dice: {metrics['macro_dice']:.1%}, ASSD: {metrics['macro_assd']:.1f}px)", (20, H + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 200), 2)
    cv2.putText(montage, "4. Error Map: Green=TP, Cyan=FP, Red=FN", (W + 20, H + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    cv2.imwrite(out_path, montage, [cv2.IMWRITE_JPEG_QUALITY, 88])

def run_evaluation(model, dataloader, device, split_name="Val", out_dir=None):
    model.eval()
    records = []
    latencies = []
    
    p40_diag_dir = os.path.join(out_dir, 'patient_40_diagnostics') if out_dir else None
    
    print(f"\n🚀 Running {split_name} Evaluation ({len(dataloader.dataset)} frames)...")
    
    with torch.no_grad():
        for idx, batch in enumerate(dataloader):
            pixel_values = batch['pixel_values'].to(device)
            masks = batch['mask'].numpy() # (B, H, W)
            gt_j_coords = batch['junction_coords'].numpy() # (B, 4, 2)
            gt_j_vis = batch['junction_vis'].numpy() # (B, 4)
            filenames = batch['filename']
            is_p40s = batch['is_patient_40']
            
            t0 = time.time()
            outputs = model(pixel_values)
            latencies.append((time.time() - t0) * 1000.0)
            
            pred_masks_logits = outputs['masks_queries_logits'] # (B, 100, H/4, W/4)
            pred_cls_logits = outputs['class_queries_logits']   # (B, 100, 5)
            pred_j_coords_t = outputs['pred_junction_coords'].float().cpu().numpy() # (B, 4, 2)
            
            for b in range(pixel_values.shape[0]):
                pred_map = rasterize_class_map(pred_masks_logits[b], pred_cls_logits[b])
                
                m = evaluate_frame_metrics(
                    pred_map=pred_map,
                    target_map=masks[b],
                    pred_j_coords=pred_j_coords_t[b],
                    gt_j_coords=gt_j_coords[b],
                    gt_j_vis=gt_j_vis[b]
                )
                m['filename'] = filenames[b]
                m['is_patient_40'] = bool(is_p40s[b])
                records.append(m)
                
                # Render 4-panel diagnostic for Patient 40 only
                if is_p40s[b] and p40_diag_dir:
                    diag_path = os.path.join(p40_diag_dir, f"{Path(filenames[b]).stem}_diag.jpg")
                    render_patient_40_diagnostic(
                        orig_rgb_norm=pixel_values[b].float().cpu().numpy(),
                        gt_mask=masks[b],
                        pred_map=pred_map,
                        pred_j_coords=pred_j_coords_t[b],
                        gt_j_coords=gt_j_coords[b],
                        gt_j_vis=gt_j_vis[b],
                        out_path=diag_path,
                        metrics=m
                    )
                    
            if (idx + 1) % 20 == 0 or (idx + 1) == len(dataloader):
                cur_dice = np.mean([r['macro_dice'] for r in records])
                cur_assd = np.mean([r['macro_assd'] for r in records])
                print(f"   [{split_name} {idx+1}/{len(dataloader)}] Macro Dice: {cur_dice:.2%} | Macro ASSD: {cur_assd:.2f}px")

    df = pd.DataFrame(records)
    
    # Compute summary
    summary = {
        'split': split_name,
        'total_frames': len(df),
        'macro_dice': float(df['macro_dice'].mean()),
        'macro_iou': float(df['macro_iou'].mean()),
        'macro_assd': float(df['macro_assd'].mean()),
        'ridge_dice': float(df['ridge_dice'].mean()),
        'sil_dice': float(df['sil_dice'].mean()),
        'falc_dice': float(df['falc_dice'].mean()),
        'fg_dice': float(df['fg_dice'].mean()),
        'mean_latency_ms': float(np.mean(latencies)),
        'fps': float(1000.0 / np.mean(latencies)),
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU/MPS"
    }
    
    # Patient 40 breakdown
    p40_df = df[df['is_patient_40']]
    if len(p40_df) > 0:
        summary['patient_40_dice'] = float(p40_df['macro_dice'].mean())
        summary['patient_40_assd'] = float(p40_df['macro_assd'].mean())
        summary['patient_40_count'] = len(p40_df)
    else:
        summary['patient_40_dice'] = 0.0
        summary['patient_40_assd'] = 80.0
        summary['patient_40_count'] = 0
        
    # Junction error breakdown
    valid_j = df['mean_j_err_px'].dropna()
    if len(valid_j) > 0:
        summary['mean_junction_err_px'] = float(valid_j.mean())
        
    return summary, df

def main():
    parser = argparse.ArgumentParser(description="Evaluate EXPERIMENT_5 Junction-Steered Mask2Former")
    parser.add_argument('--checkpoint', type=str, default=None, help="Path to best_model.pth checkpoint")
    parser.add_argument('--data_dir', type=str, default=None, help="Path to L3D dataset root")
    parser.add_argument('--out_dir', type=str, default=None, help="Output directory for results and diagnostics")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
    args = parser.parse_args()
    
    if args.out_dir is None:
        args.out_dir = os.path.join(_WORKSPACE_ROOT, 'experiments/EXPERIMENT_5/results')
    os.makedirs(args.out_dir, exist_ok=True)
    
    device = torch.device(args.device)
    print(f"🖥️ Using device: {device}")
    
    # 1. Load Model
    model = load_junction_steered_model(args.checkpoint, device=device)
    
    # 2. Validation Set
    val_dataset = L3DDataset(split='Val', data_dir=args.data_dir)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2)
    val_summary, val_df = run_evaluation(model, val_loader, device, split_name="Val", out_dir=args.out_dir)
    
    val_csv_path = os.path.join(args.out_dir, 'val_predictions.csv')
    val_df.to_csv(val_csv_path, index=False)
    print(f"💾 Saved Validation predictions: {val_csv_path}")
    
    # 3. Test Set
    test_dataset = L3DDataset(split='Test', data_dir=args.data_dir)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2)
    test_summary, test_df = run_evaluation(model, test_loader, device, split_name="Test", out_dir=args.out_dir)
    
    test_csv_path = os.path.join(args.out_dir, 'test_predictions.csv')
    test_df.to_csv(test_csv_path, index=False)
    print(f"💾 Saved Test predictions: {test_csv_path}")
    
    # 4. Save JSON Summary
    full_summary = {
        'val_summary': val_summary,
        'test_summary': test_summary,
        'checkpoint_path': args.checkpoint
    }
    summary_path = os.path.join(args.out_dir, 'metrics_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(full_summary, f, indent=4)
    print(f"🎉 Complete Metrics Summary saved to: {summary_path}")

if __name__ == '__main__':
    main()


In [ ]:
# ==============================================================================
# 8. Main Training Execution
# ==============================================================================
EPOCHS = 60
BATCH_SIZE = 2
ACCUM_STEPS = 2
LR_HEAD = 1e-4
LR_BACKBONE = 1e-5

train_dataset = L3DDataset(split="Train", data_dir=DATA_ROOT)
val_dataset = L3DDataset(split="Val", data_dir=DATA_ROOT)
test_dataset = L3DDataset(split="Test", data_dir=DATA_ROOT)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_fn_l3d)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn_l3d)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn_l3d)

model = JunctionSteeredMask2Former(num_labels=4).to(device)
criterion = JunctionSteeredLoss(lambda_m2f=1.0, lambda_coord=5.0, lambda_vis=1.0)
optimizer, scheduler = build_optimizer_and_scheduler(model, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD, epochs=EPOCHS)

print("🚀 Starting 60-epoch training on Kaggle GPU...")
best_val_dice = 0.0
training_log = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    losses = []
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        mask_labels = [m.to(device) for m in batch["mask_labels"]]
        class_labels = [c.to(device) for c in batch["class_labels"]]
        gt_j_coords = batch["junction_coords"].to(device)
        gt_j_vis = batch["junction_vis"].to(device)
        
        outputs = model(pixel_values, mask_labels=mask_labels, class_labels=class_labels)
        total_loss, loss_dict = criterion(outputs, gt_j_coords, gt_j_vis)
        (total_loss / ACCUM_STEPS).backward()
        losses.append(loss_dict["loss"])
        
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            optimizer.zero_grad()
            
    scheduler.step()
    
    # Validation evaluation
    val_summary, val_df = run_evaluation(model, val_loader, device, split_name="Val", out_dir=OUT_DIR)
    vdice = val_summary["macro_dice"]
    vassd = val_summary["macro_assd"]
    p40dice = val_summary["patient_40_dice"]
    
    print(f"Epoch [{epoch}/{EPOCHS}] Train Loss: {np.mean(losses):.4f} | Val Dice: {vdice:.2%} | ASSD: {vassd:.2f}px | P40 Dice: {p40dice:.2%} ({time.time()-t0:.1f}s)")
    
    if vdice > best_val_dice:
        best_val_dice = vdice
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_model.pth"))
        print(f"⭐ New best model saved: {best_val_dice:.2%}")

# Final Test Set Evaluation
print("
🏁 Running Final Test Set Evaluation...")
model.load_state_dict(torch.load(os.path.join(OUT_DIR, "best_model.pth")))
test_summary, test_df = run_evaluation(model, test_loader, device, split_name="Test", out_dir=OUT_DIR)
test_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)
val_df.to_csv(os.path.join(OUT_DIR, "val_predictions.csv"), index=False)

full_metrics = {"val_summary": val_summary, "test_summary": test_summary}
with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
    json.dump(full_metrics, f, indent=4)

# Zip results for download
!cd /kaggle/working && zip -r results_exp5.zip results_exp5
print("🎉 All done! Download /kaggle/working/results_exp5.zip")